# 66 — Documentary Adjudication Patch (empty-doc-safe)

This version can still build a documentary-only baseline even when the
fused documentary table exists but contains only zero-strength rows.

In [ ]:
import os, json, hashlib, platform, sys, re
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        if path.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(path), "status": "empty_file"}
        df = pd.read_csv(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest: dict, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

In [ ]:
step = "66_documentary_adjudication_patch"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

doc_cfg = load_yaml(CONFIGS / "documentary_sources.yml")
run_cfg = load_yaml(CONFIGS / "run.yml")
adjud_cfg = doc_cfg.get("documentary_adjudication", {})

adjud, adjud_meta = safe_read_csv(OUTPUTS / "56_site_adjudication" / "site_adjudication.csv")
docs, docs_meta = safe_read_csv(OUTPUTS / "65_documentary_evidence_fusion" / "documentary_evidence_site_year.csv")

sites = pd.DataFrame(run_cfg.get("sites", []))
if not sites.empty:
    sites["site_id"] = sites.get("id", pd.Series(dtype=str)).astype(str)
    sites["site_name"] = sites.get("name", pd.Series(dtype=str)).astype(str)
    if "role" not in sites.columns:
        sites["role"] = "unknown"
else:
    sites = pd.DataFrame(columns=["site_id","site_name","role"])

if docs.empty:
    docs = sites[["site_id","site_name"]].copy() if not sites.empty else pd.DataFrame(columns=["site_id","site_name"])
    docs["report_year"] = pd.Timestamp.utcnow().year
    docs["documentary_evidence_score"] = 0.0
    docs["documentary_exceedance_rows"] = 0
    docs["documentary_noncompliance_rows"] = 0
    docs["documentary_report_count"] = 0
    docs_meta = {"path": str(OUTPUTS / "65_documentary_evidence_fusion" / "documentary_evidence_site_year.csv"), "status": "synthetic_zero_baseline"}

docs_max = docs.groupby("site_id", as_index=False).agg(
    documentary_evidence_score=("documentary_evidence_score", "max"),
    documentary_exceedance_rows=("documentary_exceedance_rows", "max"),
    documentary_noncompliance_rows=("documentary_noncompliance_rows", "max"),
    documentary_report_count=("documentary_report_count", "max"),
    latest_report_year=("report_year", "max"),
)

standalone_mode = adjud.empty

if standalone_mode:
    if sites.empty:
        work = docs_max.copy()
        work["site_name"] = work["site_id"]
        work["role"] = "unknown"
    else:
        work = sites[["site_id", "site_name", "role"]].copy()
        work = work.merge(docs_max, on="site_id", how="left")
    for c in ["documentary_evidence_score","documentary_exceedance_rows","documentary_noncompliance_rows","documentary_report_count"]:
        if c not in work.columns:
            work[c] = 0
        work[c] = pd.to_numeric(work[c], errors="coerce").fillna(0)

    report_norm_den = float(adjud_cfg.get("report_count_norm_denominator", 5))
    work["integrity_score"] = np.clip(work["documentary_report_count"] / report_norm_den, 0, 1).round(3)
    work["readiness_score"] = np.where(work["documentary_report_count"] > 0, 1.0, 0.0)
    work["signal_score"] = work["documentary_evidence_score"].round(3)
    work["verdict"] = np.where(work["documentary_report_count"] > 0, "documentary_only_review", "not_ready")
    work["source_mode"] = "standalone_documentary"
else:
    work = adjud.copy()
    key_col = "site_id" if "site_id" in work.columns else ("site_name" if "site_name" in work.columns else None)
    if key_col != "site_id":
        work["site_id"] = work[key_col].map(slugify)
    work = work.merge(docs_max, on="site_id", how="left")
    work["source_mode"] = "patched_full_pipeline"

for c in ["documentary_evidence_score","documentary_exceedance_rows","documentary_noncompliance_rows","documentary_report_count"]:
    if c not in work.columns:
        work[c] = 0
    work[c] = pd.to_numeric(work[c], errors="coerce").fillna(0)

report_norm_den = float(adjud_cfg.get("report_count_norm_denominator", 5))
documentary_support_threshold = float(adjud_cfg.get("documentary_supported_threshold", 1.0))

work["integrity_score_patched"] = (
    pd.to_numeric(work.get("integrity_score", 0), errors="coerce").fillna(0).astype(float) * 0.7
    + np.clip(work["documentary_report_count"] / report_norm_den, 0, 1) * 0.3
).round(3)

work["signal_score_patched"] = (
    pd.to_numeric(work.get("signal_score", 0), errors="coerce").fillna(0).astype(float)
    + work["documentary_evidence_score"]
).round(3)

def patched_verdict(row):
    if row["documentary_exceedance_rows"] > 0 or row["documentary_noncompliance_rows"] > 0:
        return "priority_review"
    if row["documentary_evidence_score"] >= documentary_support_threshold and row.get("verdict", "") != "not_ready":
        return "documentary_supported_review"
    if row.get("source_mode") == "standalone_documentary" and row["documentary_report_count"] > 0:
        return "documentary_only_review"
    return row.get("verdict", "not_ready")

work["verdict_patched"] = work.apply(patched_verdict, axis=1)
work.to_csv(out / "site_adjudication_patched.csv", index=False)

summary = {
    "standalone_mode": bool(standalone_mode),
    "rows": int(len(work)),
    "priority_review_sites": int((work["verdict_patched"] == "priority_review").sum()),
    "documentary_supported_sites": int((work["verdict_patched"] == "documentary_supported_review").sum()),
    "documentary_only_review_sites": int((work["verdict_patched"] == "documentary_only_review").sum()),
}
write_json(out / "documentary_adjudication_patch_summary.json", summary)

manifest = manifest_base(step, [CONFIGS / "documentary_sources.yml", CONFIGS / "run.yml"])
manifest["inputs"] = {"site_adjudication": adjud_meta, "documentary_evidence_site_year": docs_meta}
add_artifact(manifest, out / "site_adjudication_patched.csv")
add_artifact(manifest, out / "documentary_adjudication_patch_summary.json")
manifest["summary"] = summary
write_json(out / "manifest.json", manifest)

print(summary)
display(work.head(20))